# Siem Reap Soil Parameters — 02 Data Cleaning

The rules of `settings.toml` are applied to every target, and the samples used by the models are saved. Each rule is counted, so the thesis can report exactly what was removed and why.

**What you'll learn:**
- which records become model samples, and how many each rule removes
- how the soil type input is coded (Clay 1 / Sand 0)
- which boreholes are development boreholes (in V3's spatial folds) and which are V3's external holdout
- the depth trend of each parameter after cleaning (what a depth-only model already knows)

**Rules (decided before any modelling):**

| Rule | Setting |
|---|---|
| Area | Downtown boreholes (as V3) |
| Depth | middle of the record in 0-15 m |
| Record length | at most 3 m (longer records are not point values) |
| SPT-N | values above 60 removed (refusal; as Khamjan et al. 2026); **no** 3 SD rule |
| Value range (other parameters) | inside the target's `min`/`max` in settings (obvious errors) |
| Soil subset | Su and Eu: clay samples only; E′: sand samples only |
| Outliers (all parameters except SPT-N) | values beyond mean ± 3 SD dropped (as Khamjan et al. 2026) |
| Soil input | logged USCS: Clay (CL, CH) = 1, Sand (SC, SM, SP) = 0; other symbols left out |
| Elevation input | ground elevation − top depth of the record |
| Sets | V3's 11 external-holdout boreholes (hash-checked) kept apart; development = boreholes in V3's folds |

No gap filling: these are point values. SPT refusals and "non-plastic" entries were blanked in the CSV export, so they are missing, not zero.

**Run time:** under 1 minute.

## 1. Import Libraries

In [ ]:
import json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import utils as U

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
SOIL_COLORS = {"Clay": "#5B8C3A", "Sand": "#E5B74A", "other": "#9E9E9E", "missing": "#D0D0D0"}

S = U.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
P, O = U.processed_dir(S), U.out_dir(S, "02_data_cleaning")

## 2. Load the Dataset and V3's Sets

In [ ]:
records = U.load_downtown(S)
holdout = U.load_v3_holdout(S)
folds = U.load_folds(S, U.block_sizes(S)[0])
print(f"Downtown records: {len(records)} | boreholes: {records.BH_Code.nunique()}")
print(f"V3 development boreholes (in folds): {len(folds)} | V3 external holdout: {len(holdout)} (lock checked)")

## 3. Apply the Rules to Every Target

In [ ]:
# One file of samples per target (data/processed/, not in git)
rows, info = [], {}
for t in S["cv"]["targets"]:
    smp, inf = U.make_samples(records, t, S)
    smp["set"] = U.assign_sets(smp, holdout.index, folds)
    smp.to_csv(P / f"samples_{t}.csv", index=False)
    info[t] = inf
    for s, g in smp.groupby("set"):
        rows.append({"target": t, "set": s, "boreholes": g.BH_Code.nunique(), "samples": len(g),
                     **g.soil_name.value_counts().sort_index().to_dict(),
                     "mean": g.y.mean(), "sd": g.y.std(), "min": g.y.min(), "median": g.y.median(), "max": g.y.max()})
steps = pd.DataFrame(info).T
steps.to_csv(O / "cleaning_steps.csv")
steps

**Before → after:** `records_with_value` = values in the Downtown file; `samples` = values the models use. The other columns count what each rule removed.

In [ ]:
drop_cols = [c for c in steps.columns if c.startswith("dropped_") and c not in ("dropped_small_groups",)]
d = steps[drop_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
ax = d.plot(kind="barh", stacked=True, figsize=(10, 5), colormap="tab10")
ax.set_xlabel("records removed"); ax.set_title("What each rule removed, per target"); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(O / "removed_by_rule.png", dpi=200); plt.show()

## 4. Samples per Target and Set

In [ ]:
counts = pd.DataFrame(rows)
counts.to_csv(O / "sample_counts.csv", index=False)
counts.round(2)

**Key observations:** `development` samples are used in notebooks 03-04; `external` (V3's holdout) is kept for a once-only test later; `no_fold` boreholes have no V3 fold (they had no Clay/Sand cell in V3) and are left out.

## 5. Depth Profiles after Cleaning (development set)
Mean and interquartile range per 1 m slice. The **Depth** reference model predicts exactly this curve; the spatial models must beat it (notebook 04).

In [ ]:
ts = S["cv"]["targets"]
ncol = 5; nrow = int(np.ceil(len(ts) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 4.2 * nrow), sharey=True)
prof = []
for ax, t in zip(axes.ravel(), ts):
    d = pd.read_csv(P / f"samples_{t}.csv").query("set == 'development'")
    g = d.groupby(np.floor(d.depth_m)).y
    p = pd.DataFrame({"n": g.size(), "mean": g.mean(), "q25": g.quantile(0.25), "q75": g.quantile(0.75)})
    prof.append(p.assign(target=t))
    ax.fill_betweenx(p.index + 0.5, p.q25, p.q75, alpha=0.3)
    ax.plot(p["mean"], p.index + 0.5, marker="o", ms=3)
    ax.set_title(t); ax.set_xlabel(S["targets"][t]["unit"])
for ax in axes.ravel()[len(ts):]:
    ax.axis("off")
axes.ravel()[0].invert_yaxis()
for ax in axes[:, 0]:
    ax.set_ylabel("depth (m)")
plt.tight_layout(); plt.savefig(O / "profiles_after_cleaning.png", dpi=200); plt.show()
pd.concat(prof).rename_axis("depth_from_m").to_csv(O / "profiles_after_cleaning.csv")

## 6. Value by Soil Group (development set)
Large differences between Clay and Sand mean the soil input alone explains much of the variation; the **SoilDepth** reference (mean per soil group and depth slice) measures exactly that.

In [ ]:
by_soil = pd.concat([pd.read_csv(P / f"samples_{t}.csv").query("set == 'development'")
                     .groupby("soil_name").y.describe().assign(target=t) for t in ts]).reset_index()
by_soil.to_csv(O / "values_by_soil_group.csv", index=False)
by_soil.round(2)

In [ ]:
summary = {"targets": ts, "soil_input": S["soil_input"]["mode"], "elevation": S["samples"]["elevation"],
           "cleaning": info, "holdout_boreholes": int(len(holdout))}
(O / "cleaning_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 03 (model building).")